<img loading="lazy" src="https://cdn.jsdelivr.net/gh/devicons/devicon@latest/icons/python/python-original.svg" width="40" height="40"/> <img src="https://cdn.jsdelivr.net/gh/devicons/devicon@latest/icons/pandas/pandas-original-wordmark.svg" width="40" height="40"/>

---

#**<font color=#85d338 size="6"> Import libraries**

In [ ]:
# Bibliotecas
import pandas as pd
import requests
from io import BytesIO
import zipfile
import re

#**<font color=#85d338 size="6"> 1. Coleta de dados**

In [1]:
# ==========================================================
# ETL - Download e Importação dos Dados da ANATEL
# Autor: Gabriel Prata
# Projeto: Banda Larga Fixa
# ==========================================================

# CONFIGURAÇÕES
# URL do arquivo ZIP contendo os dados abertos da Anatel
URL = "https://www.anatel.gov.br/dadosabertos/paineis_de_dados/acessos/acessos_banda_larga_fixa.zip"

# Anos que voce quer coletar
ANOS = [2019, 2020, 2025, 2026]

# -------------------------------------------------------

def mapear_arquivos_do_zip(zip_file):
    padrao = re.compile(r"Acessos_Banda_Larga_Fixa_(\d{4})(?:-(\d{4}))?\.csv")
    mapa = {}

    for nome in zip_file.namelist():
        m = padrao.search(nome)
        if not m:
            continue

        inicio = int(m.group(1))
        fim = int(m.group(2)) if m.group(2) else inicio

        for ano in range(inicio, fim + 1):
            mapa[ano] = nome

    return mapa

def obter_conjunto_dados(url, anos):
    print("="*60)
    print("Download Conjunto de Dados da ANATEL...")
    print("="*60)

    response = requests.get(url)
    response.raise_for_status()

    zip_file = zipfile.ZipFile(BytesIO(response.content))

    mapa_arquivos = mapear_arquivos_do_zip(zip_file)

    dataframes_por_ano = {}
    cache_arquivos = {} # evita ler o arquivo agrupado mais de uma vez

    for ano in anos:
        nome = mapa_arquivos.get(ano)

        if nome is None:
            print(f"Nenhum arquivo encontrado para o ano {ano}.")
            continue

        if nome not in cache_arquivos:
            with zip_file.open(nome) as f:
                cache_arquivos[nome] = pd.read_csv(f, sep=";", low_memory=False)

        df = cache_arquivos[nome]
        dataframes_por_ano[ano] = df

        # Filtra só as linhas do ano correspondente
        df_filtrado = df[df["Ano"].astype(str) == str(ano)].copy()

        # Cria automaticamente o dataframe com os parâmetros de ano solicitado
        globals()[f"df_{ano}"] = df_filtrado
        print(f"Carregando {nome} (cobre o ano {ano})...")
        print(f"✔ df_{ano}: {df_filtrado.shape[0]:,} linhas x {df_filtrado.shape[1]} colunas\n")

    return dataframes_por_ano

# EXECUÇÃO
dados = obter_conjunto_dados(URL, ANOS)

Download Conjunto de Dados da ANATEL...
Carregando Acessos_Banda_Larga_Fixa_2019-2020.csv (cobre o ano 2019)...
✔ df_2019: 1,763,090 linhas x 13 colunas

Carregando Acessos_Banda_Larga_Fixa_2019-2020.csv (cobre o ano 2020)...
✔ df_2020: 1,882,014 linhas x 13 colunas

Carregando Acessos_Banda_Larga_Fixa_2025.csv (cobre o ano 2025)...
✔ df_2025: 8,156,040 linhas x 16 colunas

Carregando Acessos_Banda_Larga_Fixa_2026.csv (cobre o ano 2026)...
✔ df_2026: 3,413,808 linhas x 16 colunas



#**<font color=#85d338 size="6"> 2. Validações**

###**<font color=#85d338> 2.1. Resumo**

In [6]:
for ano, df in dados.items():
    print(f"\n=== {ano} ===")
    print("shape:", df.shape)


=== 2019 ===
shape: (3645104, 13)

=== 2020 ===
shape: (3645104, 13)

=== 2025 ===
shape: (8156040, 16)

=== 2026 ===
shape: (3413808, 16)


###**<font color=#85d338> 2.2. Missings**

In [7]:
for ano, df in dados.items():
    print(f"\n=== {ano} ===")
    print("nulos por coluna:")
    print(df.isna().sum())


=== 2019 ===
nulos por coluna:
Ano                      0
Mês                      0
Grupo Econômico          0
Empresa                  0
CNPJ                     0
Porte da Prestadora      0
UF                       0
Município                0
Código IBGE Município    0
Faixa de Velocidade      0
Tecnologia               0
Meio de Acesso           0
Acessos                  0
dtype: int64

=== 2020 ===
nulos por coluna:
Ano                      0
Mês                      0
Grupo Econômico          0
Empresa                  0
CNPJ                     0
Porte da Prestadora      0
UF                       0
Município                0
Código IBGE Município    0
Faixa de Velocidade      0
Tecnologia               0
Meio de Acesso           0
Acessos                  0
dtype: int64

=== 2025 ===
nulos por coluna:
Ano                      0
Mês                      0
Grupo Econômico          0
Empresa                  0
CNPJ                     0
Porte da Prestadora      0
UF            

###**<font color=#85d338> 2.3. Info**

In [10]:
for ano, df in dados.items():
    print(f"\n================ {ano} ================")
    print("Informacoes:")
    print(df.info())


================ 2019 ================
Informacoes:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3645104 entries, 0 to 3645103
Data columns (total 13 columns):
 #   Column                 Dtype 
---  ------                 ----- 
 0   Ano                    int64 
 1   Mês                    int64 
 2   Grupo Econômico        object
 3   Empresa                object
 4   CNPJ                   int64 
 5   Porte da Prestadora    object
 6   UF                     object
 7   Município              object
 8   Código IBGE Município  int64 
 9   Faixa de Velocidade    object
 10  Tecnologia             object
 11  Meio de Acesso         object
 12  Acessos                int64 
dtypes: int64(5), object(8)
memory usage: 361.5+ MB
None

================ 2020 ================
Informacoes:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3645104 entries, 0 to 3645103
Data columns (total 13 columns):
 #   Column                 Dtype 
---  ------                 ----- 
 0   Ano        